# 🧠 Step 3: Non-Stationary Signal Analysis & Channel Selection

In multi-channel biomedical signal processing (such as **19-channel EEG** or **12-lead ECG**), recordings are inherently **non-stationary** (their statistical properties—mean, variance, and frequency content—change dynamically over time).

### Objective:
Instead of utilizing all 12 channels for classification, we apply **Inter-Class vs. Intra-Class Discriminability Analysis** (Fisher Criterion / ANOVA F-Score) on non-stationary segment features (Mean & Variance) to identify and select the **3 to 4 most informative, unique channels**.

---
### Mathematical Framework (Adapted from 19-Channel EEG Channel Selection):
For each channel $c \in \{1, \dots, 12\}$:
1. **Non-Stationary Segmentation**: Compute local mean $\mu_{i,c}(t)$ and local variance $\sigma^2_{i,c}(t)$ across sliding temporal windows.
2. **Intra-Class Variance ($S_W$)**: Within each diagnostic class $k$, compute dispersion among patients of the same disease:
   $$S_W(c) = \sum_{k=1}^K \frac{1}{N_k} \sum_{i \in C_k} (x_{i,c} - \bar{x}_{k,c})^2$$
3. **Inter-Class Variance ($S_B$)**: Measure separation of disease class means from the global population mean:
   $$S_B(c) = \sum_{k=1}^K \frac{N_k}{N} (\bar{x}_{k,c} - \bar{x}_c)^2$$
4. **Fisher Discriminability Ratio ($J(c)$)**:
   $$J(c) = \frac{S_B(c)}{S_W(c)}$$
Channels with the highest $J(c)$ provide maximum separation between cardiac diseases with minimal intra-class noise.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb

# Set paths (supports both ./dataset and ../data/raw/ptbxl)
DATA_DIR = "./dataset" if os.path.exists("./dataset/ptbxl_database.csv") else "../data/raw/ptbxl"
DB_CSV = os.path.join(DATA_DIR, "ptbxl_database.csv")
SCP_CSV = os.path.join(DATA_DIR, "scp_statements.csv")

LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

print(f"Data Directory: {DATA_DIR}")
df = pd.read_csv(DB_CSV, index_col="ecg_id")
print(f"Loaded metadata: {len(df)} records.")

In [ ]:
# Parse diagnostic superclasses for each record
import ast

scp_df = pd.read_csv(SCP_CSV, index_col=0)
diag_map = scp_df[scp_df["diagnostic"] == 1]["diagnostic_class"].to_dict()

def get_primary_superclass(scp_str):
    try:
        d = ast.literal_eval(scp_str)
        classes = [diag_map[k] for k in d.keys() if k in diag_map]
        return classes[0] if classes else "NORM"
    except Exception:
        return "NORM"

df["primary_class"] = df["scp_codes"].apply(get_primary_superclass)
print("Diagnostic Class Distribution:")
print(df["primary_class"].value_counts())

In [ ]:
# Load signals (uses 100 Hz or 500 Hz cache if available, or loads sample subset)
cache_100 = os.path.join(DATA_DIR, "signals_cache_100hz.npy")
cache_500 = os.path.join(DATA_DIR, "signals_cache_500hz.npy")

if os.path.exists(cache_500):
    print("Loading 500 Hz binary cache...")
    signals = np.load(cache_500)
elif os.path.exists(cache_100):
    print("Loading 100 Hz binary cache...")
    signals = np.load(cache_100)
else:
    print("Loading first 500 records from disk...")
    sub_df = df.iloc[:500]
    sigs = []
    fn_col = "filename_hr" if "filename_hr" in df.columns else "filename_lr"
    for fn in sub_df[fn_col]:
        rec, _ = wfdb.rdsamp(os.path.join(DATA_DIR, fn))
        sigs.append(rec)
    signals = np.array(sigs)
    df = sub_df

print(f"Signals matrix shape: {signals.shape} (Records x TimeSteps x Leads)")

In [ ]:
# ==========================================================================
# NON-STATIONARY SIGNAL ANALYSIS: TIME-SEGMENTED VARIANCE & MEAN
# ==========================================================================
# Split 10-second ECG signals into non-stationary temporal windows (e.g. 5 segments)
num_windows = 5
windows = np.array_split(signals, num_windows, axis=1)

# Calculate variance and mean across temporal windows per record and channel
win_vars = np.array([np.var(w, axis=1) for w in windows]) # (num_windows, N, 12)
win_means = np.array([np.mean(w, axis=1) for w in windows]) # (num_windows, N, 12)

# Non-stationary descriptor: temporal variance of local variance (energy fluctuation)
non_stat_feature = np.var(win_vars, axis=0) + np.var(win_means, axis=0) # Shape: (N, 12)

labels = df["primary_class"].values[:len(signals)]

# Compute Inter-Class (SB) and Intra-Class (SW) variance for each channel
channel_metrics = []
K = len(SUPERCLASSES)
N_total = len(labels)

for ch_idx, lead in enumerate(LEAD_NAMES):
    f_ch = non_stat_feature[:, ch_idx]
    global_mean = np.mean(f_ch)
    
    sb = 0.0 # Inter-Class Variance
    sw = 0.0 # Intra-Class Variance
    
    for c in SUPERCLASSES:
        mask = (labels == c)
        n_c = np.sum(mask)
        if n_c > 0:
            c_vals = f_ch[mask]
            c_mean = np.mean(c_vals)
            sb += n_c * ((c_mean - global_mean) ** 2)
            sw += np.sum((c_vals - c_mean) ** 2)
            
    sb = sb / (K - 1)
    sw = sw / (N_total - K)
    f_ratio = sb / (sw + 1e-10)
    
    channel_metrics.append({
        "Lead": lead,
        "Channel_Index": ch_idx,
        "Inter_Class_Var_SB": sb,
        "Intra_Class_Var_SW": sw,
        "Discriminability_Ratio": f_ratio
    })

results_df = pd.DataFrame(channel_metrics).sort_values(by="Discriminability_Ratio", ascending=False).reset_index(drop=True)
results_df.index += 1
results_df.index.name = "Rank"

print("\n" + "="*65)
print("  12-LEAD CHANNEL SELECTION BY INTER/INTRA CLASS DISCRIMINABILITY")
print("="*65)
print(results_df.to_string())

In [ ]:
# Select Top 3 to 4 Unique Channels
top_4_channels = results_df.head(4)["Lead"].tolist()
top_4_indices = results_df.head(4)["Channel_Index"].tolist()

print(f"\n>>> SELECTED TOP 4 UNIQUE CHANNELS: {top_4_channels}")
print(f">>> Channel Indices: {top_4_indices}")

In [ ]:
# Visual Plot 1: Channel Discriminability Ranking Bar Chart
plt.figure(figsize=(12, 6))
colors = ["#2a9d8f" if l in top_4_channels else "#a8dadc" for l in results_df["Lead"]]
bars = plt.bar(results_df["Lead"], results_df["Discriminability_Ratio"], color=colors, edgecolor="#1d3557", linewidth=1.2)

plt.axhline(results_df.iloc[3]["Discriminability_Ratio"], color="#e63946", linestyle="--", linewidth=1.5, label="Selection Threshold (Top 4)")
plt.title("Channel Selection via Non-Stationary Inter/Intra Class Ratio (Fisher Score)", fontsize=13, fontweight="bold")
plt.xlabel("ECG Lead (Channel)", fontsize=11, fontweight="bold")
plt.ylabel("Discriminability Ratio (SB / SW)", fontsize=11, fontweight="bold")
plt.legend(loc="upper right", fontsize=10)
plt.grid(True, linestyle=":", alpha=0.6)

# Annotate values
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., h + 0.1, f"{h:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

os.makedirs("artifacts/eda", exist_ok=True)
plt.savefig("artifacts/eda/channel_selection_ranking.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Visual Plot 2: Waveform of the Selected 4 Channels for a Sample Record
sample_record = signals[0] # (TimeSteps, 12)
time_axis = np.arange(len(sample_record)) / (500 if len(sample_record) == 5000 else 100)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f"Selected Top 4 Unique ECG Channels for Classification: {top_4_channels}", fontsize=14, fontweight="bold")

for ax, lead_name, ch_idx in zip(axes, top_4_channels, top_4_indices):
    ax.plot(time_axis, sample_record[:, ch_idx], color="#1d3557", linewidth=1.3)
    ax.set_ylabel(f"{lead_name}\n(mV)", fontsize=10, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)

axes[-1].set_xlabel("Time (seconds)", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("artifacts/eda/selected_channels_ecg.png", dpi=300, bbox_inches="tight")
plt.show()
print("Channel Selection Step 3 Completed Successfully!")